# Riemannian Score-MD — Implementation Exploration

This notebook lets you interactively verify the three layers of the implementation:

1. **ShapeManifold** — w^delta geometry: distances, log maps, tangent spaces
2. **ManifoldVP** — VP-SDE forward process with ambient noising
3. **ManifoldEulerMaruyama** — Brownian motion on the manifold

Uses the fast synthetic n=10 protein by default. Toggle `FAST_MODE = False` in section 0 to load real adenylate kinase (n=214, requires MDAnalysis + DCD file).

In [ ]:
import sys
from pathlib import Path

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from manifold.pointcloud_jax import ShapeManifold
from diffusion.manifold_sde import ManifoldVP
from diffusion.manifold_solvers import ManifoldEulerMaruyama

print('JAX devices:', jax.devices())

## 0. Load data

In [ ]:
FAST_MODE = True   # set False for real AK (n=214, requires MDAnalysis)

DATA_PATH = ROOT.parent / 'diepeveen2024' / 'data' / 'molecular_dynamics' / '4ake'

def load_frame(fast=FAST_MODE, frame_idx=0):
    if not fast:
        try:
            import MDAnalysis as mda
            u = mda.Universe(
                str(DATA_PATH / 'adk4ake.psf'),
                str(DATA_PATH / 'dims0001_fit-core.dcd')
            )
            u.trajectory[frame_idx]
            ca = u.select_atoms('name CA')
            coords = ca.positions.astype(np.float32)
            return jnp.array(coords[None, None])   # (1, 1, 214, 3)
        except (ImportError, FileNotFoundError) as e:
            print(f'[warn] Could not load AK: {e}. Falling back to synthetic.')
    rng = np.random.default_rng(42)
    x = rng.standard_normal((1, 1, 10, 3)).astype(np.float32) * 10.0
    return jnp.array(x)

def load_all_frames(max_frames=None):
    """Load multiple AK frames for dataset-level checks."""
    try:
        import MDAnalysis as mda
        u = mda.Universe(
            str(DATA_PATH / 'adk4ake.psf'),
            str(DATA_PATH / 'dims0001_fit-core.dcd')
        )
        frames = []
        for i, _ in enumerate(u.trajectory):
            if max_frames and i >= max_frames:
                break
            ca = u.select_atoms('name CA')
            frames.append(ca.positions.astype(np.float32))
        return jnp.array(np.stack(frames))[:, None]   # (F, 1, n, d)
    except (ImportError, FileNotFoundError):
        rng = np.random.default_rng(0)
        return jnp.array(rng.standard_normal((20, 1, 10, 3)).astype(np.float32) * 10.0)

x0 = load_frame()
n, d = x0.shape[2], x0.shape[3]
print(f'Loaded: shape={x0.shape}, n={n} Cα atoms, d={d}')

manifold = ShapeManifold(dim=d, numpoints=n, alpha=1.0, base=np.array(x0[0, 0]))
sde = ManifoldVP(manifold)
print(f'Manifold dim = {manifold.manifold_dimension}, vert dim = {manifold.vert_space_dimension}')

## 1. ShapeManifold — w^delta geometry

### 1a. Distances and the metric tensor

In [ ]:
# Distance between x0 and a perturbed copy
rng = np.random.default_rng(1)
eps_vals = [0.001, 0.01, 0.1, 0.5, 1.0, 2.0]

print(f"{'eps':>8}  {'w^delta dist':>14}  {'Euclidean dist':>16}")
print('-' * 42)
for eps in eps_vals:
    noise = jnp.array(rng.standard_normal(x0.shape).astype(np.float32))
    y = x0 + eps * noise
    y = manifold.center_mpoint(y)  # centre to remove translation
    
    wdelta = float(manifold.s_distance(x0, y)[0, 0, 0])
    eucl = float(jnp.linalg.norm(x0 - y))
    print(f'{eps:>8.3f}  {wdelta:>14.6f}  {eucl:>16.6f}')

In [ ]:
# Metric tensor H: eigenvalue spectrum tells us about the geometry
H = manifold.metric_tensor(x0, asmatrix=True)   # (1, 1, nd, nd)
evals = jnp.linalg.eigvalsh(H[0, 0])            # (nd,)

vert_dim = manifold.vert_space_dimension
horiz_dim = manifold.manifold_dimension

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].semilogy(np.array(evals), '.', markersize=3)
axes[0].axvline(vert_dim, color='red', linestyle='--', alpha=0.7, label=f'vert/horiz split (k={vert_dim})')
axes[0].set_xlabel('Eigenvalue index')
axes[0].set_ylabel('Eigenvalue (log scale)')
axes[0].set_title('Metric tensor eigenspectrum')
axes[0].legend()

# Zoom into horizontal block
horiz_evals = np.array(evals[vert_dim:])
axes[1].hist(horiz_evals, bins=30, edgecolor='white', linewidth=0.5)
axes[1].set_xlabel('Eigenvalue')
axes[1].set_title(f'Horizontal eigenvalues ({horiz_dim} DOF)')

plt.tight_layout()
plt.show()

print(f'Vertical block (should be ~0): min={float(evals[:vert_dim].min()):.2e}, max={float(evals[:vert_dim].max()):.2e}')
print(f'Horizontal block: min={float(evals[vert_dim:].min()):.4f}, max={float(evals[vert_dim:].max()):.4f}, cond={float(evals[-1]/evals[vert_dim]):.1f}')

### 1b. Log map: direction from x_t back to x_0

In [ ]:
# Check that s_log inverts ambient displacement (the key property for ambient noising)
# For x_t = x0 + sigma*v_h, we expect s_log(x_t, x0) ≈ -sigma*v_h

rng_jax = jax.random.PRNGKey(42)
v = jax.random.normal(rng_jax, x0.shape)
v_hp = manifold.horizontal_projection_tvector(x0, v[:, :, None])  # (1,1,1,n,d)
nrm = manifold.norm(x0, v_hp)                                     # (1,1,1)
v_h_unit = v_hp / jnp.maximum(nrm, 1e-8)[:, :, :, None, None]    # unit g-norm

sigma_vals = [0.05, 0.1, 0.2, 0.4, 0.8]
print(f'Checking s_log(x0 + sigma*v_h, x0) ≈ -sigma*v_h')
print(f"{'sigma':>8}  {'cos(log, -tangent)':>20}  {'||log|| / sigma':>16}  {'verdict':>8}")
print('-' * 58)

for sigma in sigma_vals:
    tangent = float(sigma) * v_h_unit[:, :, 0]   # (1, 1, n, d)
    x_t = x0 + tangent
    log = manifold.s_log(x_t, x0)[0, 0, 0]       # (n, d)
    
    t_flat = tangent[0, 0].flatten()
    l_flat = log.flatten()
    
    cos = float(jnp.dot(l_flat, -t_flat) / (jnp.linalg.norm(l_flat) * jnp.linalg.norm(t_flat) + 1e-12))
    mag_ratio = float(jnp.linalg.norm(l_flat)) / float(sigma)
    verdict = 'OK' if cos > 0.99 else 'WARN'
    print(f'{sigma:>8.2f}  {cos:>20.6f}  {mag_ratio:>16.4f}  {verdict:>8}')

### 1c. Horizontal projection: vertical components are zeroed

In [ ]:
# The horizontal projection should remove the rotational (vertical) component
# Vertical tangent vectors are infinitesimal rotations: V = x @ Omega^T where Omega is antisymmetric
# After horizontal projection, V should map to zero

# Build a purely vertical tangent vector (infinitesimal rotation)
Omega = jnp.array([[[0., -1., 0.],
                     [1.,  0., 0.],
                     [0.,  0., 0.]]])  # (1, d, d) antisymmetric
x_centred = manifold.center_mpoint(x0)  # (1, 1, n, d)
V_vert = jnp.einsum('NMnd,Mde->NMne', x_centred, Omega)  # (1, 1, n, d) — infinitesimal rotation

V_projected = manifold.horizontal_projection_tvector(x0, V_vert[:, :, None])  # (1,1,1,n,d)
norm_before = float(jnp.linalg.norm(V_vert))
norm_after  = float(jnp.linalg.norm(V_projected))

print(f'Infinitesimal rotation (vertical tangent):')
print(f'  ||V_vert|| before projection: {norm_before:.6f}')
print(f'  ||V_vert|| after projection:  {norm_after:.2e}  (should be ~0)')

# A horizontal vector should be unchanged by projection (up to normalisation)
v_rand = jax.random.normal(jax.random.PRNGKey(7), x0.shape)
v_h = manifold.horizontal_projection_tvector(x0, v_rand[:, :, None])         # project once
v_hh = manifold.horizontal_projection_tvector(x0, v_h)                        # project again
idempotent_err = float(jnp.linalg.norm(v_h - v_hh))
print(f'\nIdempotency check (project twice = project once):')
print(f'  ||P(Pv) - Pv|| = {idempotent_err:.2e}  (should be ~0)')

### 1d. Geodesic interpolation

In [ ]:
# Interpolate between x0 and a perturbed x1, check distances are monotone
rng_np = np.random.default_rng(5)
noise = jnp.array(rng_np.standard_normal(x0.shape).astype(np.float32))
x1 = manifold.center_mpoint(x0 + 0.5 * noise)

tau_vals = jnp.array([0.0, 0.25, 0.5, 0.75, 1.0])
z = manifold.s_geodesic(x0, x1, tau_vals)  # (1, 5, n, d)

d_from_x0 = [float(manifold.s_distance(x0, z[:, i:i+1])[0, 0, 0]) for i in range(5)]
d_from_x1 = [float(manifold.s_distance(x1, z[:, i:i+1])[0, 0, 0]) for i in range(5)]
total_d    = float(manifold.s_distance(x0, x1)[0, 0, 0])

print(f'Total w^delta distance d(x0,x1) = {total_d:.6f}')
print(f"{'tau':>6}  {'d(z,x0)':>12}  {'d(z,x1)':>12}  {'d(z,x0)+d(z,x1)':>18}")
print('-' * 52)
for i, tau in enumerate([0.0, 0.25, 0.5, 0.75, 1.0]):
    print(f'{tau:>6.2f}  {d_from_x0[i]:>12.6f}  {d_from_x1[i]:>12.6f}  {d_from_x0[i]+d_from_x1[i]:>18.6f}')

# Plot distances along geodesic
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot([0, 0.25, 0.5, 0.75, 1.0], d_from_x0, 'o-', label='d(z(τ), x0)')
ax.plot([0, 0.25, 0.5, 0.75, 1.0], d_from_x1, 's-', label='d(z(τ), x1)')
ax.axhline(total_d / 2, color='gray', linestyle='--', alpha=0.5, label='d(x0,x1)/2')
ax.set_xlabel('τ (geodesic parameter)')
ax.set_ylabel('w^delta distance')
ax.set_title('Distance along geodesic interpolation')
ax.legend()
plt.tight_layout()
plt.show()

## 2. ManifoldVP — forward process

### 2a. VP schedule: sigma(t) and alpha(t)

In [ ]:
ts = jnp.linspace(0, 1, 200)
sigmas = jax.vmap(sde.sigma)(ts)
alphas = jax.vmap(sde.alpha)(ts)
betas  = jax.vmap(sde.beta)(ts)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].plot(ts, alphas, label='alpha(t)')
axes[0].set_title('Mean decay alpha(t) = exp(log_alpha)')
axes[0].set_xlabel('t'); axes[0].set_ylabel('alpha')

axes[1].plot(ts, sigmas, label='sigma(t)', color='orange')
axes[1].set_title('Noise level sigma(t) = sqrt(1 - alpha^2)')
axes[1].set_xlabel('t'); axes[1].set_ylabel('sigma')

axes[2].plot(ts, betas, label='beta(t)', color='green')
axes[2].set_title(f'beta(t): linear from {sde.beta_min} to {sde.beta_max}')
axes[2].set_xlabel('t'); axes[2].set_ylabel('beta')

for ax in axes:
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 2b. Noising: x_t = x_0 + alpha(t)*sigma(t)*v_h

In [ ]:
# Show how noising progressively corrupts the conformation
t_vals = [0.0, 0.1, 0.2, 0.4, 0.6, 0.8, 1.0]
rng = jax.random.PRNGKey(0)

print('Forward process: x_t = x_0 + alpha(t)*sigma(t)*v_h')
print(f"{'t':>6}  {'sigma(t)':>10}  {'alpha(t)':>10}  {'||x_t-x0||_F':>14}  {'w^delta(x_t,x0)':>18}")
print('-' * 65)

xt_list = []
for t_val in t_vals:
    t = jnp.array(t_val)
    rng, subkey = jax.random.split(rng)
    x_t, v_h_unit, sigma_t = sde.marginal_prob(x0, t, subkey)
    
    sigma_v = float(sde.sigma(t))
    alpha_v = float(sde.alpha(t))
    eucl_dist = float(jnp.linalg.norm(x_t - x0))
    
    # w^delta distance (may be NaN if x_t is very far off manifold at large t)
    try:
        wdist = float(manifold.s_distance(x_t, x0)[0, 0, 0])
        wdist_str = f'{wdist:>18.4f}'
    except:
                wdist_str = '             N/A'
    
    print(f'{t_val:>6.2f}  {sigma_v:>10.4f}  {alpha_v:>10.4f}  {eucl_dist:>14.4f}  {wdist_str}')
    xt_list.append(x_t)

In [ ]:
# Visualise: pairwise distance matrix at different noise levels
# For small t, it should look similar to t=0; for large t it should be random

fig, axes = plt.subplots(1, len(t_vals), figsize=(3*len(t_vals), 3))
pw0 = np.array(jnp.sqrt(jnp.maximum(manifold.pairwise_distances(x0)[0, 0], 0)))

for i, (t_val, x_t) in enumerate(zip(t_vals, xt_list)):
    pw = np.array(jnp.sqrt(jnp.maximum(manifold.pairwise_distances(x_t)[0, 0], 0)))
    vmax = pw0.max()
    im = axes[i].imshow(pw, cmap='viridis', vmin=0, vmax=vmax)
    axes[i].set_title(f't={t_val:.1f}', fontsize=9)
    axes[i].axis('off')

axes[0].set_title('t=0 (clean)', fontsize=9)
fig.suptitle('Pairwise Cα distance matrix at different noise levels', y=1.02)
plt.tight_layout()
plt.show()

### 2c. Score target: s_true = -s_log(x_t, x_0) / sigma(t)^2

In [ ]:
# Check that the score target points in the right direction:
# s_true should point FROM x_t BACK TOWARD x_0
# i.e. x_t + s_true * sigma^2 ≈ x_0  (in ambient space for small sigma)

rng = jax.random.PRNGKey(10)
print('Score target consistency: x_t + sigma^2 * s_true ≈ x_0')
print(f"{'sigma':>8}  {'||x_t-x0||_F':>14}  {'||reconstruction-x0||_F':>24}  {'ratio':>8}")
print('-' * 60)

for sigma in [0.05, 0.1, 0.2, 0.4, 0.6]:
    # Find t s.t. sigma(t) ≈ sigma (binary search)
    # Just pick t directly by inverting sigma numerically
    import scipy.optimize as opt
    try:
        def f(tt):
            return float(sde.sigma(jnp.array(float(tt)))) - sigma
        t_val = opt.brentq(f, 0.0, 1.0)
    except:
        t_val = sigma  # fallback
    t = jnp.array(float(t_val))
    
    rng, subkey = jax.random.split(rng)
    x_t, v_h_unit, sigma_t = sde.marginal_prob(x0, t, subkey)
    s_true = sde.score_target(x_t, x0, t)   # (1, 1, 1, n, d)
    
    # Reconstruct: x_t + sigma^2 * s_true should ≈ x_0
    reconstruction = x_t + float(sigma_t)**2 * s_true[:, :, 0]
    err_before = float(jnp.linalg.norm(x_t - x0))
    err_after  = float(jnp.linalg.norm(reconstruction - x0))
    print(f'{sigma:>8.2f}  {err_before:>14.4f}  {err_after:>24.4f}  {err_after/max(err_before,1e-10):>8.3f}')

## 3. ManifoldEulerMaruyama — Brownian motion

### 3a. Short trajectory: does it stay near the manifold?

In [ ]:
solver = ManifoldEulerMaruyama(sde, manifold, mode='forward')
dt = 1e-4
n_steps = 50
rng = jax.random.PRNGKey(100)

x = x0.copy()
Rg0 = float(jnp.sqrt(jnp.trace(manifold.gyration_matrix(x)[0, 0]) / n))
mean_pw0 = float(jnp.mean(jnp.sqrt(jnp.maximum(manifold.pairwise_distances(x)[0, 0], 0))))

history = {'Rg': [Rg0], 'mean_pw': [mean_pw0], 't': [0.0]}

for step in range(n_steps):
    t = jnp.array(step * dt)
    x, rng = solver.step(x, t, dt, rng)
    Rg = float(jnp.sqrt(jnp.trace(manifold.gyration_matrix(x)[0, 0]) / n))
    mean_pw = float(jnp.mean(jnp.sqrt(jnp.maximum(manifold.pairwise_distances(x)[0, 0], 0))))
    history['Rg'].append(Rg)
    history['mean_pw'].append(mean_pw)
    history['t'].append((step + 1) * dt)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history['t'], history['Rg'])
axes[0].axhline(Rg0, color='red', linestyle='--', label=f'Initial Rg={Rg0:.3f}')
axes[0].fill_between(history['t'],
                      Rg0 * 0.9, Rg0 * 1.1, alpha=0.15, color='red', label='±10% band')
axes[0].set_xlabel('t'); axes[0].set_ylabel('Gyration radius Rg')
axes[0].set_title('Gyration radius during Brownian motion')
axes[0].legend()

axes[1].plot(history['t'], history['mean_pw'])
axes[1].axhline(mean_pw0, color='red', linestyle='--', label=f'Initial={mean_pw0:.3f}')
axes[1].fill_between(history['t'],
                      mean_pw0 * 0.95, mean_pw0 * 1.05, alpha=0.15, color='red', label='±5% band')
axes[1].set_xlabel('t'); axes[1].set_ylabel('Mean pairwise distance')
axes[1].set_title('Pairwise distances during Brownian motion')
axes[1].legend()

plt.tight_layout()
plt.show()

Rg_drift = abs(history['Rg'][-1] - Rg0) / Rg0
pw_drift  = abs(history['mean_pw'][-1] - mean_pw0) / mean_pw0
print(f'Final Rg drift: {Rg_drift*100:.2f}%  (test threshold: 10%)')
print(f'Final pairwise drift: {pw_drift*100:.2f}%  (test threshold: 5%)')

### 3b. Noise scaling: ||v_h||_g should equal g(t)*sqrt(dt)

In [ ]:
# Verify diffusion coefficient at a few t values
dt_test = 1e-4
n_traj = 30

print(f'Diffusion coefficient check: ||noise||_g vs g(t)*sqrt(dt)')
print(f"{'t':>6}  {'g(t)':>8}  {'expected':>12}  {'measured':>12}  {'rel_err':>10}")
print('-' * 54)

for t_val in [0.0, 0.2, 0.5, 0.8, 1.0]:
    t = jnp.array(t_val)
    g = float(sde.diffusion_coeff(t))
    expected = g * float(jnp.sqrt(dt_test))
    
    norms = []
    for i in range(n_traj):
        rng_i = jax.random.PRNGKey(300 + i)
        v_raw = jax.random.normal(rng_i, x0.shape)
        v_h = manifold.horizontal_projection_tvector(x0, v_raw[:, :, None])[:, :, 0]
        nrm = manifold.norm(x0, v_h[:, :, None])[:, :, 0]
        v_h_unit = v_h / jnp.maximum(nrm[:, :, None, None], 1e-8)
        v_scaled = g * jnp.sqrt(dt_test) * v_h_unit
        norms.append(float(manifold.norm(x0, v_scaled[:, :, None])[0, 0, 0]))
    
    measured = float(np.mean(norms))
    rel_err = abs(measured - expected) / expected
    print(f'{t_val:>6.2f}  {g:>8.4f}  {expected:>12.6f}  {measured:>12.6f}  {rel_err:>10.4f}')

## 4. Coordinate normalisation utility

Verify `normalize` / `denormalize` roundtrip.

In [ ]:
x_norm, scale = manifold.normalize(x0)
x_recovered = manifold.denormalize(x_norm, scale)

Rg_orig = float(manifold.gyration_scale(x0)[0, 0])
Rg_norm = float(manifold.gyration_scale(x_norm)[0, 0])
roundtrip_err = float(jnp.max(jnp.abs(x_recovered - x0)))

print(f'Gyration scale before normalisation: {Rg_orig:.4f}')
print(f'Gyration scale after  normalisation: {Rg_norm:.4f}  (should be ≈1)')
print(f'Roundtrip error max|x_recovered - x0|: {roundtrip_err:.2e}  (should be ~0)')

## 5. End-to-end: forward process then score target

Sanity check the full DSM training pipeline on a mini-batch.

In [ ]:
# Simulate a training step: for each sample in a batch, compute (x_t, s_true)
# then check that s_true has unit g-norm (up to the 1/sigma^2 scaling)

batch_size = 8
rng = jax.random.PRNGKey(42)

print('Mini-batch forward pass simulation')
print(f"{'sample':>8}  {'t':>6}  {'sigma':>8}  {'||s_true||_F':>14}  {'1/sigma^2':>12}")
print('-' * 54)

rng_np = np.random.default_rng(0)
t_samples = rng_np.uniform(0.01, 0.99, size=batch_size)

for i, t_val in enumerate(t_samples):
    t = jnp.array(float(t_val))
    rng, subkey = jax.random.split(rng)
    x_t, v_h_unit, sigma_t = sde.marginal_prob(x0, t, subkey)
    s_true = sde.score_target(x_t, x0, t)  # (1, 1, 1, n, d)
    
    s_norm = float(jnp.linalg.norm(s_true[0, 0, 0]))
    expected = 1.0 / float(sigma_t)**2  # unit g-norm tangent / sigma^2
    
    print(f'{i:>8}  {t_val:>6.3f}  {float(sigma_t):>8.4f}  {s_norm:>14.4f}  {expected:>12.4f}')

## 6. (Optional) Load full adenylate kinase and check dataset statistics

Requires `FAST_MODE = False` above **and** MDAnalysis + DCD file.

In [ ]:
if not FAST_MODE:
    frames = load_all_frames(max_frames=102)   # all 102 AK frames
    F = frames.shape[0]
    print(f'Loaded {F} frames, shape={frames.shape}')
    
    # Pairwise w^delta distance matrix between all frames
    # (slow for n=214 — uses s_distance which is O(n^2))
    print('Computing pairwise distance matrix...')
    D = np.zeros((F, F))
    for i in range(F):
        for j in range(i+1, F):
            d = float(manifold.s_distance(frames[i:i+1], frames[j:j+1])[0, 0, 0])
            D[i, j] = D[j, i] = d
    
    plt.figure(figsize=(8, 6))
    plt.imshow(D, cmap='viridis')
    plt.colorbar(label='w^delta distance')
    plt.title(f'Pairwise w^delta distances: {F} AK frames')
    plt.tight_layout()
    plt.show()
    
    print(f'Min inter-frame distance: {D[D>0].min():.4f}')
    print(f'Max inter-frame distance: {D.max():.4f}')
    print(f'Mean inter-frame distance: {D[D>0].mean():.4f}')
else:
    print('Skipped (FAST_MODE=True). Set FAST_MODE=False to run on real AK data.')